# Phase 1 — Architecture 4 run (Kaggle 2× T4 **or** Colab 1× T4)

Same harness as the Prefix-LM and encoder-decoder runs. **Yours to write/decide (rule 17.5):** `model.py` (cell `%%writefile model.py`) and every `TODO` in the CONFIG cell.

| | Kaggle (preferred: 2 GPUs) | Colab (1 GPU) |
|---|---|---|
| Settings | GPU T4 ×2, Internet On, secret `HF_TOKEN` | Runtime → T4 GPU, secret `HF_TOKEN` (key icon) |
| Inputs | attach dataset `prefixlm-run1` (token files + tokenizer found at any depth) | Drive folder `MyDrive/arch4/`: put `tokenizer/vocab.json` + `tokenizer/merges.txt` there (and optionally `tokens/` to skip pre-tokenizing) |
| Checkpoints | `/kaggle/working/run/` | `MyDrive/arch4/run/` (a disconnect resumes automatically: just Run all again) |

**Clock:** set `FINISH_BY_IST` and `MAX_SESSION_HOURS` in CONFIG. Training stops in time for local eval + decoding sweep + test generation to finish by then.

**Order:** fill `model.py` + CONFIG → run cells 1–7b interactively until **ALL CHECKS PASSED** and the speed line looks right → (Kaggle) stop session → *Save & Run All*; (Colab) *Run all*.
Result: `submission.zip` (validated) + `arch4_best.pt` / `arch4_final.pt`.

In [ ]:
# 1 — platform, session clock, deps, secrets
import time, os, json, glob, shutil, datetime as dt
SESSION_START = time.time()
ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = (not ON_KAGGLE) and os.path.exists("/content")
assert ON_KAGGLE or ON_COLAB, "run this notebook on Kaggle or Colab"
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/arch4"
    os.makedirs(DRIVE_DIR, exist_ok=True)
    WORK = "/content/work"
    INPUT_ROOTS = [DRIVE_DIR]
else:
    DRIVE_DIR = None
    WORK = "/kaggle/working"
    INPUT_ROOTS = ["/kaggle/input", "/tmp/prev"]
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
!pip install -q ijson bert-score
import torch
NGPU = torch.cuda.device_count()
assert NGPU >= 1, "no GPU: Kaggle -> Accelerator GPU T4 x2; Colab -> Runtime -> Change runtime type -> T4 GPU"
print("platform:", "Kaggle" if ON_KAGGLE else "Colab", "| GPUs:", NGPU, [torch.cuda.get_device_name(i) for i in range(NGPU)])


try:
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    else:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN secret not available:", e)

## 2 — Configuration
Every `TODO` is your decision. The notebook refuses to start until none are left.

**Held equal to the earlier runs (controlled comparison):** tokenizer, token files, data mix, `ctx_len`, tokens/step, LR schedule + optimizer settings, validation windows and `val_prefix_*`, decoding settings. **Changed:** the architecture (`model.py`, `MODEL`) and `prefix_mode`.
If you change any held-equal value, say so in the report.

In [ ]:
# 3 — CONFIG  (values marked [=arch2] are kept identical to the earlier runs)
TODO = "TODO"
RUN_NAME = "arch4"

# ---- clock: everything (training + local eval + sweep + test zip) finishes by FINISH_BY_IST
FINISH_BY_IST = "22:00"
MAX_SESSION_HOURS = 10.5
IST = dt.timezone(dt.timedelta(hours=5, minutes=30))
_now = dt.datetime.now(IST); _h, _m = map(int, FINISH_BY_IST.split(":"))
_fin = _now.replace(hour=_h, minute=_m, second=0, microsecond=0)
if _fin <= _now:
    _fin += dt.timedelta(days=1)
SESSION_HOURS = min(MAX_SESSION_HOURS, (_fin.timestamp() - SESSION_START) / 3600)
print(f"session budget {SESSION_HOURS:.2f} h -> done by "
      f"{dt.datetime.fromtimestamp(SESSION_START + SESSION_HOURS * 3600, IST):%H:%M} IST")


def _glob_inputs(pattern):
    hits = set()
    for r in INPUT_ROOTS:
        hits.update(glob.glob(f"{r}/**/{pattern}", recursive=True))
    return sorted(hits)


# ---- data + output locations
if ON_KAGGLE:
    _metas = [m for m in _glob_inputs("meta.json") if glob.glob(os.path.join(os.path.dirname(m), "train_*.bin"))]
    TOKEN_DIR = os.path.dirname(_metas[0]) if _metas else f"{WORK}/tokens"
    TOKEN_CACHE = None
    OUT_DIR = f"{WORK}/run"
else:
    TOKEN_DIR = "/content/tokens"
    TOKEN_CACHE = f"{DRIVE_DIR}/tokens"
    OUT_DIR = f"{DRIVE_DIR}/run"
print("token_dir:", TOKEN_DIR, "| out_dir:", OUT_DIR)

_ckpts = sorted(set(_glob_inputs(f"{RUN_NAME}_step*.pt") + glob.glob(f"{OUT_DIR}/{RUN_NAME}_step*.pt")),
                key=lambda q: int(q.split("_step")[-1][:-3]))
_prev_ckpt = _ckpts[-1] if _ckpts else None
MUST_RESUME = False
assert _prev_ckpt or not MUST_RESUME, f"no {RUN_NAME}_step*.pt found — attach/upload the checkpoint (or set MUST_RESUME=False)"
print("resume checkpoint:", _prev_ckpt or "none (fresh run)")
if _prev_ckpt and ON_COLAB:
    print("  -> training will CONTINUE from it. For a fresh run, delete MyDrive/arch4/run first.")

RUN = dict(
    run_name      = RUN_NAME,
    session_start = SESSION_START,
    session_hours = SESSION_HOURS,
    reserve_minutes_after_train = 75 if NGPU >= 2 else 110,
    token_dir     = TOKEN_DIR,
    tokenizer_dir = f"{WORK}/tokenizer",
    out_dir       = OUT_DIR,
    resume_dirs   = [os.path.dirname(_prev_ckpt)] if _prev_ckpt else [],
    val_windows_per_source = 48,
    eval_ckpt_overhead_frac = 0.05,
    smoke_test    = False,
)

MODEL = dict(
    d_model=768, n_layers=9, n_heads=6, d_ff=3072, value_factor=2,
    gamma_start_exp=5.0, rope_base=10000.0, dropout=0.0, tie_embeddings=True,
)

SOURCES = ["supreme_court", "high_courts", "acts", "tribunals"]
MIX = dict(supreme_court=0.40, high_courts=0.30, acts=0.20, tribunals=0.10)

TRAIN = dict(
    ctx_len=1024,
    micro_batch=8,
    grad_accum=16 // NGPU,
    lr_max=3e-4, lr_min=3e-5, warmup_steps=1000,
    schedule="cosine",
    decay_frac=None,
    weight_decay=0.1, wd_exclude_1d=True,
    beta1=0.9, beta2=0.95, grad_clip=1.0,
    prefix_mode="none",
    prefix_frac_min=0.0, prefix_frac_max=0.0,
    val_prefix_mode="fixed", val_prefix_frac=0.75,
    mix=MIX,
    max_tokens=None,
    amp_dtype="fp16",
    compile=False, grad_ckpt=False, find_unused=False,
    log_every_steps=20,
    val_every_min=15,
    ckpt_every_min=20,
)

GEN = dict(
    checkpoint="best",
    max_new_tokens=256,
    batch_size=32,
    temperature=0,
    top_k=0, top_p=1.0, rep_penalty=1.0,
    no_repeat_ngram=4, ngram_scope="generated",
    seed=0,
)

TOTAL_BUDGET = 1_500_000_000
PRETOK = dict(
    budgets={s: int(TOTAL_BUDGET * MIX[s]) for s in SOURCES},
    keep_every=1, val_permille=5, local_pairs_per_source=100,
)

CFG = dict(run=RUN, model=MODEL, train=TRAIN, gen=GEN, pretok=PRETOK)


def find_todos(obj, path=""):
    if obj == TODO: return [path]
    if isinstance(obj, dict): return sum((find_todos(v, f"{path}.{k}") for k, v in obj.items()), [])
    return []


missing = find_todos(CFG)
if missing:
    print("Still TODO:", *missing, sep="\n  ")
else:
    json.dump(CFG, open("run_config.json", "w"), indent=1)
    print("run_config.json written")
assert TRAIN["prefix_mode"] in ("none", "fixed", "random", "batch_random", TODO), "bad prefix_mode"

In [ ]:
# 4 — tokenizer (your BPE from the earlier runs)
os.makedirs(RUN["tokenizer_dir"], exist_ok=True)
for fn in ["vocab.json", "merges.txt"]:
    dst = os.path.join(RUN["tokenizer_dir"], fn)
    if not os.path.exists(dst):
        hits = _glob_inputs(fn)
        assert hits, (f"{fn} not found under {INPUT_ROOTS} — Kaggle: attach the dataset holding your tokenizer; "
                      f"Colab: put it in MyDrive/arch4/tokenizer/")
        shutil.copy(hits[0], dst); print("tokenizer file:", hits[0])
from tokenizers import ByteLevelBPETokenizer
_tok = ByteLevelBPETokenizer(vocab=f"{RUN['tokenizer_dir']}/vocab.json", merges=f"{RUN['tokenizer_dir']}/merges.txt")
print("vocab:", _tok.get_vocab_size(), "| </s> id:", _tok.token_to_id("</s>"))

In [ ]:
%%writefile pretokenize.py
"""pretokenize.py — one-time data prep for all Phase-1 runs (ancillary code, not model code)."""
import argparse, json, os, random, time, zlib

import numpy as np


def parse_kv(items, cast=str):
    out = {}
    for it in items or []:
        k, v = it.split("=", 1)
        out[k.strip()] = cast(v.replace("_", "")) if cast is int else cast(v)
    return out


def is_heldout(text, permille):

    return zlib.crc32(text[:2000].encode("utf-8", "ignore")) % 1000 < permille


def load_tokenizer(tok_dir):
    from tokenizers import ByteLevelBPETokenizer
    tok = ByteLevelBPETokenizer(
        vocab=os.path.join(tok_dir, "vocab.json"),
        merges=os.path.join(tok_dir, "merges.txt"),
    )
    return tok


class BinWriter:

    def __init__(self, path):
        self.f = open(path, "wb")
        self.n = 0

    def write(self, ids_list):
        if not ids_list:
            return
        arr = np.fromiter((t for ids in ids_list for t in ids), dtype=np.uint16)
        arr.tofile(self.f)
        self.n += arr.size

    def close(self):
        self.f.close()


def process_source(name, path, budget, tok, eos_id, args, local_pairs, rng):
    import ijson

    tr = BinWriter(os.path.join(args.out, f"train_{name}.bin"))
    va = BinWriter(os.path.join(args.out, f"val_{name}.bin"))
    stats = dict(docs_seen=0, docs_train=0, docs_val=0, docs_short=0, docs_skipped_stride=0)

    batch_tr, batch_va = [], []
    t0 = time.time()

    def flush():
        for batch, writer in ((batch_tr, tr), (batch_va, va)):
            if batch:
                enc = tok.encode_batch(batch)
                writer.write([e.ids + [eos_id] for e in enc])
                batch.clear()

    with open(path, "rb") as f:
        for i, rec in enumerate(ijson.items(f, "item")):
            text = rec.get(args.text_key, "") if isinstance(rec, dict) else ""
            if not isinstance(text, str) or len(text) < args.min_chars:
                stats["docs_short"] += 1
                continue
            if args.keep_every > 1 and i % args.keep_every != 0:
                stats["docs_skipped_stride"] += 1
                continue
            stats["docs_seen"] += 1

            if is_heldout(text, args.val_permille):
                stats["docs_val"] += 1
                if va.n < args.val_budget_per_source:
                    batch_va.append(text)
                words = text.split()
                need = args.prompt_words + args.cont_words
                if len(words) >= need and len(local_pairs[name]) < args.local_pairs_per_source:
                    s = rng.randint(0, len(words) - need)
                    local_pairs[name].append((
                        " ".join(words[s:s + args.prompt_words]),
                        " ".join(words[s + args.prompt_words:s + need]),
                    ))
            else:
                stats["docs_train"] += 1
                batch_tr.append(text)

            if len(batch_tr) >= args.batch_docs or len(batch_va) >= args.batch_docs:
                flush()
                if stats["docs_seen"] % (args.batch_docs * 20) == 0:
                    rate = tr.n / max(1e-9, time.time() - t0)
                    print(f"  [{name}] train={tr.n/1e6:8.1f}M tok  val={va.n/1e6:6.2f}M  "
                          f"docs={stats['docs_seen']:,}  {rate/1e3:.0f}K tok/s", flush=True)
            if tr.n >= budget:
                break
    flush()
    tr.close(); va.close()
    stats.update(train_tokens=tr.n, val_tokens=va.n, seconds=round(time.time() - t0, 1))
    print(f"  [{name}] DONE train={tr.n/1e6:.1f}M val={va.n/1e6:.2f}M "
          f"in {stats['seconds']/60:.1f} min", flush=True)
    return stats


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--tok_dir", required=True, help="folder with vocab.json + merges.txt")
    ap.add_argument("--out", required=True)
    ap.add_argument("--file", action="append", required=True, help="name=/path/to/file.json")
    ap.add_argument("--budget", action="append", required=True,
                    help="name=max_train_tokens (your data-mix choice)")
    ap.add_argument("--text_key", default="text")
    ap.add_argument("--min_chars", type=int, default=100)
    ap.add_argument("--keep_every", type=int, default=1,
                    help="keep every k-th doc to spread sampling across a file (1 = all)")
    ap.add_argument("--val_permille", type=int, default=5, help="held-out docs per 1000")
    ap.add_argument("--val_budget_per_source", type=int, default=2_000_000)
    ap.add_argument("--local_pairs_per_source", type=int, default=100)
    ap.add_argument("--prompt_words", type=int, default=512)
    ap.add_argument("--cont_words", type=int, default=128)
    ap.add_argument("--batch_docs", type=int, default=256)
    ap.add_argument("--eos_token", default="</s>")
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    files = parse_kv(args.file)
    budgets = parse_kv(args.budget, int)
    missing = set(files) - set(budgets)
    if missing:
        raise SystemExit(f"No --budget given for: {sorted(missing)}")
    os.makedirs(args.out, exist_ok=True)

    tok = load_tokenizer(args.tok_dir)
    vocab = tok.get_vocab_size()
    if vocab > 65535:
        raise SystemExit(f"vocab {vocab} does not fit uint16")
    eos_id = tok.token_to_id(args.eos_token)
    if eos_id is None:
        raise SystemExit(f"EOS token {args.eos_token!r} not in vocab")

    rng = random.Random(args.seed)
    local_pairs = {n: [] for n in files}
    meta = dict(settings=vars(args), vocab_size=vocab, eos_id=eos_id, dtype="uint16", sources={})

    for name, path in files.items():
        print(f"Processing {name}: {path} (budget {budgets[name]/1e6:.0f}M train tokens)")
        meta["sources"][name] = process_source(name, path, budgets[name], tok, eos_id,
                                               args, local_pairs, rng)

    test, ref, origin = {}, {}, {}
    k = 1
    for name, pairs in local_pairs.items():
        for p, r in pairs:
            test[str(k)] = {"text": p}
            ref[str(k)] = {"text": r}
            origin[str(k)] = name
            k += 1
    for fn, obj in (("local_test.json", test), ("local_ref.json", ref), ("local_origin.json", origin)):
        with open(os.path.join(args.out, fn), "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False)
    meta["local_pairs"] = {n: len(p) for n, p in local_pairs.items()}
    with open(os.path.join(args.out, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    tot = sum(s["train_tokens"] for s in meta["sources"].values())
    print(f"\nAll done. Train tokens: {tot/1e6:.1f}M | local test pairs: {len(test)}")
    print(f"Files in {args.out}: train_*.bin, val_*.bin, local_test.json, local_ref.json, meta.json")


if __name__ == "__main__":
    main()

In [ ]:
# 6 — token files: attached (Kaggle) / Drive cache (Colab) / pre-tokenize
from huggingface_hub import hf_hub_download
REPO = "Exploration-Lab/CS-781-Capstone"
TEST_PATH = hf_hub_download(REPO, "Phase_1_Pretraining/Test/test.json", repo_type="dataset")

if os.path.exists(os.path.join(TOKEN_DIR, "meta.json")):
    print("Token files ready in", TOKEN_DIR)
elif TOKEN_CACHE and os.path.exists(os.path.join(TOKEN_CACHE, "meta.json")):
    t0 = time.time(); shutil.copytree(TOKEN_CACHE, TOKEN_DIR, dirs_exist_ok=True)
    print(f"copied token files from Drive in {(time.time()-t0)/60:.1f} min")
else:
    args = f"--tok_dir {RUN['tokenizer_dir']} --out {TOKEN_DIR} "
    args += f"--keep_every {PRETOK['keep_every']} --val_permille {PRETOK['val_permille']} "
    args += f"--local_pairs_per_source {PRETOK['local_pairs_per_source']} "
    for name, budget in PRETOK["budgets"].items():
        path = hf_hub_download(REPO, f"Phase_1_Pretraining/Train/train_{name}.json", repo_type="dataset")
        args += f"--file {name}={path} --budget {name}={int(budget)} "
    t0 = time.time()
    print("running: python pretokenize.py", args)
    !python pretokenize.py {args} 2>&1
    print(f"pre-tokenization took {(time.time()-t0)/60:.1f} min")
    if TOKEN_CACHE and os.path.exists(os.path.join(TOKEN_DIR, "meta.json")):
        shutil.copytree(TOKEN_DIR, TOKEN_CACHE, dirs_exist_ok=True); print("token files cached on Drive:", TOKEN_CACHE)
assert os.path.exists(os.path.join(TOKEN_DIR, "meta.json")), (
    "NO TOKEN FILES in " + TOKEN_DIR + " — the real error is printed above this line")
print("token files OK:", sorted(os.listdir(TOKEN_DIR)))

In [ ]:
# 6b — data + length checks
import numpy as np
_meta = json.load(open(os.path.join(TOKEN_DIR, "meta.json")))
src_tok = {s: v["train_tokens"] for s, v in _meta["sources"].items()}
print({s: f"{n/1e6:.0f}M" for s, n in src_tok.items()})
for s in SOURCES:
    print(f"  {s:15s} 1 pass = {src_tok[s] / max(1e-9, MIX[s]) / 1e9:.2f}B total training tokens at this mix")
loc_r = [len(_tok.encode(v["text"]).ids) for v in json.load(open(f"{TOKEN_DIR}/local_ref.json")).values()]
tst_p = [len(_tok.encode(v["text"]).ids) for v in json.load(open(TEST_PATH, encoding="utf-8")).values()]
keep = TRAIN["ctx_len"] - GEN["max_new_tokens"]
print(f"max_new_tokens={GEN['max_new_tokens']} covers {np.mean(np.array(loc_r) <= GEN['max_new_tokens'] - 8)*100:.1f}% of 128-word continuations")
print(f"prompt window {keep} tokens: {np.mean(np.array(tst_p) > keep)*100:.1f}% of test prompts are left-trimmed")

In [ ]:
%%writefile model.py
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

DEFAULTS = dict(
    d_model=768, n_layers=9, n_heads=6, d_ff=3072, value_factor=2,
    gamma_start_exp=5.0, rope_base=10000.0, dropout=0.0, tie_embeddings=True,
)


def rope_tables(dim, max_len, base):
    inv = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float64) / dim))
    f = torch.outer(torch.arange(max_len, dtype=torch.float64), inv)
    f = torch.cat([f, f], dim=-1)
    return f.cos().float(), f.sin().float()


def apply_rope(x, cos, sin):
    h = x.shape[-1] // 2
    rot = torch.cat([-x[..., h:], x[..., :h]], dim=-1)
    return x * cos.to(x.dtype) + rot * sin.to(x.dtype)


class MultiScaleRetention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.d, self.h = c["d_model"], c["n_heads"]
        self.dk = self.d // self.h
        self.dv = self.dk * c["value_factor"]
        self.vd = self.h * self.dv
        self.proj = nn.Linear(self.d, 2 * self.d + 2 * self.vd, bias=False)
        self.out = nn.Linear(self.vd, self.d, bias=False)
        self.gn = nn.GroupNorm(self.h, self.vd, eps=1e-5)
        self.scale = self.dk ** -0.5

    def _qkvg(self, x, cos, sin):
        B, T, _ = x.shape
        q, k, v, g = self.proj(x).split([self.d, self.d, self.vd, self.vd], dim=-1)
        q = q.view(B, T, self.h, self.dk).transpose(1, 2)
        k = k.view(B, T, self.h, self.dk).transpose(1, 2)
        v = v.view(B, T, self.h, self.dv).transpose(1, 2)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin) * self.scale
        return q, k, v, g

    def _finish(self, o, g):
        B, H, T, dv = o.shape
        o = self.gn(o.transpose(1, 2).reshape(B * T, H * dv)).view(B, T, H * dv)
        return self.out(F.silu(g) * o)

    def forward(self, x, cos, sin, mask, w=None):
        q, k, v, g = self._qkvg(x, cos, sin)
        r = (q @ k.transpose(-1, -2)) * mask.to(q.dtype)


        den = r.detach().float().sum(-1, keepdim=True).abs().clamp(min=1.0)
        o = (r / den.to(r.dtype)) @ v
        y = self._finish(o, g)
        if w is None:
            return y
        with torch.autocast(device_type=x.device.type, enabled=False):
            kf, vf = k.float(), v.float()
            S = torch.einsum("bhld,bhle,hl->bhde", kf, vf, w)
            z = torch.einsum("bhld,hl->bhd", kf, w)
        return y, (S, z)

    def step(self, x, cos, sin, state, gamma, rs):
        q, k, v, g = self._qkvg(x, cos, sin)
        S, z = state
        with torch.autocast(device_type=x.device.type, enabled=False):
            q, k, v = q.float()[:, :, 0], k.float()[:, :, 0], v.float()[:, :, 0]
            S = gamma[None, :, None, None] * S + k[..., :, None] * v[..., None, :]
            z = gamma[None, :, None] * z + k
            inv = rs.rsqrt()[None, :, None]
            o = torch.einsum("bhd,bhde->bhe", q, S) * inv
            s = ((q * z).sum(-1, keepdim=True) * inv).abs().clamp(min=1.0)
            o = (o / s)[:, :, None]
        return self._finish(o, g), (S, z)


class MLP(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.fc1 = nn.Linear(c["d_model"], c["d_ff"])
        self.fc2 = nn.Linear(c["d_ff"], c["d_model"])

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class RetNetBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln1 = nn.LayerNorm(c["d_model"])
        self.msr = MultiScaleRetention(c)
        self.ln2 = nn.LayerNorm(c["d_model"])
        self.mlp = MLP(c)
        self.drop = nn.Dropout(c["dropout"])

    def forward(self, x, cos, sin, mask):
        x = x + self.drop(self.msr(self.ln1(x), cos, sin, mask))
        return x + self.drop(self.mlp(self.ln2(x)))

    def forward_state(self, x, cos, sin, mask, w):
        a, st = self.msr(self.ln1(x), cos, sin, mask, w)
        x = x + a
        return x + self.mlp(self.ln2(x)), st

    def step(self, x, cos, sin, state, gamma, rs):
        a, st = self.msr.step(self.ln1(x), cos, sin, state, gamma, rs)
        x = x + a
        return x + self.mlp(self.ln2(x)), st


class RetentionLM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        V, D, L, H = c["vocab_size"], c["d_model"], c["n_layers"], c["n_heads"]
        self.max_len = c["max_seq_len"]
        self.tok_emb = nn.Embedding(V, D)
        self.drop = nn.Dropout(c["dropout"])
        self.blocks = nn.ModuleList([RetNetBlock(c) for _ in range(L)])
        self.ln_f = nn.LayerNorm(D)
        self.head = nn.Linear(D, V, bias=False)

        gamma = 1.0 - 2.0 ** (-c["gamma_start_exp"] - torch.arange(H, dtype=torch.float64))
        log_gamma = gamma.log()
        n = torch.arange(2 * self.max_len, dtype=torch.float64)
        powers = torch.exp(log_gamma[:, None] * n[None, :])
        diff = n[:self.max_len, None] - n[None, :self.max_len]
        decay = torch.exp(log_gamma[:, None, None] * diff.clamp(min=0)) * (diff >= 0)
        decay = decay / decay.sum(-1, keepdim=True).sqrt()
        rs = (1.0 - powers * gamma[:, None]) / (1.0 - gamma[:, None])
        cos, sin = rope_tables(D // H, 2 * self.max_len, c["rope_base"])
        self._cpu = dict(mask=decay.float(), powers=powers.float(), rs=rs.float(), gamma=gamma.float(),
                         cos=cos, sin=sin)
        self._dev = {}
        self.grad_ckpt = False

        self.apply(self._init)
        for name, p in self.named_parameters():
            if name.endswith("msr.out.weight") or name.endswith("mlp.fc2.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * L))
        if c["tie_embeddings"]:
            self.head.weight = self.tok_emb.weight

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def tables(self, device):
        key = str(device)
        if key not in self._dev:
            self._dev[key] = {k: v.to(device) for k, v in self._cpu.items()}
        return self._dev[key]

    def set_gradient_checkpointing(self, flag):
        self.grad_ckpt = bool(flag)

    def forward(self, input_ids, prefix_lens=None):
        T = input_ids.shape[1]
        t = self.tables(input_ids.device)
        x = self.drop(self.tok_emb(input_ids))
        cos, sin, mask = t["cos"][:T], t["sin"][:T], t["mask"][:, :T, :T]
        for blk in self.blocks:
            if self.grad_ckpt and self.training:
                x = checkpoint(blk, x, cos, sin, mask, use_reentrant=False)
            else:
                x = blk(x, cos, sin, mask)
        return self.head(self.ln_f(x))

    @torch.no_grad()
    def prefill(self, input_ids, prefix_lens=None):
        T = input_ids.shape[1]
        t = self.tables(input_ids.device)
        x = self.tok_emb(input_ids)
        cos, sin, mask = t["cos"][:T], t["sin"][:T], t["mask"][:, :T, :T]
        w = t["powers"][:, :T].flip(-1)
        states = []
        for blk in self.blocks:
            x, st = blk.forward_state(x, cos, sin, mask, w)
            states.append(st)
        return self.head(self.ln_f(x[:, -1])), {"states": states, "pos": T}

    @torch.no_grad()
    def decode_step(self, next_ids, cache):
        n = cache["pos"]
        t = self.tables(next_ids.device)
        x = self.tok_emb(next_ids)
        cos, sin, rs = t["cos"][n:n + 1], t["sin"][n:n + 1], t["rs"][:, n]
        for i, blk in enumerate(self.blocks):
            x, cache["states"][i] = blk.step(x, cos, sin, cache["states"][i], t["gamma"], rs)
        cache["pos"] = n + 1
        return self.head(self.ln_f(x[:, -1])), cache


def build_model(model_cfg):
    return RetentionLM({**DEFAULTS, **model_cfg})

In [ ]:
%%writefile test_arch4_model.py
"""test_arch4_model.py — correctness checks for YOUR architecture-4 model.py (test code only, no architecture)."""
import json, sys

import torch
import torch.nn.functional as F

import model as model_lib

cfg = json.load(open(sys.argv[1])) if len(sys.argv) > 1 else {"model": {}, "train": {"ctx_len": 256}}
mcfg = dict(cfg["model"])
mcfg.setdefault("vocab_size", 32000)
CTX = int(cfg["train"]["ctx_len"])
mcfg.setdefault("max_seq_len", CTX)
dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.manual_seed(0)

m = model_lib.build_model(mcfg).to(dev).float().eval()
V = mcfg["vocab_size"]
T = min(96, CTX)
P = (3 * T) // 4
B = 3
TOL = 1e-4
ok = True


def report(name, passed, detail=""):
    global ok
    ok &= bool(passed)
    print(f"[{'PASS' if passed else 'FAIL'}] {name} {detail}", flush=True)


def run(ids, p):
    with torch.no_grad():
        out = m(ids.to(dev), torch.full((ids.shape[0],), p, device=dev)).float()
    return out, ids.shape[1] - out.shape[1]


ids = torch.randint(5, V, (B, T))

# 1 — shape
out, off = run(ids, P)
okshape = tuple(out.shape) in [(B, T, V), (B, T - P + 1, V)]
report("1 output shape", okshape, f"got {tuple(out.shape)}, want {(B, T, V)} or {(B, T - P + 1, V)}")
if not okshape:
    sys.exit(1)
first = (P - 1) - off

# 2 — no leak from later continuation tokens
leaks = []
for j in [P, P + 5, T - 1]:
    ids2 = ids.clone(); ids2[:, j] = (ids2[:, j] + 7) % V
    o2, _ = run(ids2, P)
    upto = j - off
    if upto > 0:
        d = (o2[:, :upto] - out[:, :upto]).abs().max().item()
        if d > TOL:
            leaks.append((j, round(d, 6)))
report("2 no leak from later tokens", not leaks, f"leaks (pos, max diff): {leaks}" if leaks else "")

# 3 — the prompt matters
ids3 = ids.clone(); ids3[:, P - 10] = (ids3[:, P - 10] + 7) % V
o3, _ = run(ids3, P)
d3 = (o3[:, first] - out[:, first]).abs().max().item()
report("3 prediction depends on the prompt", d3 > TOL, f"(changed by {d3:.2e})")

# 4 — rows independent
diffs = []
for b in range(B):
    ob, _ = run(ids[b:b + 1], P)
    diffs.append((ob[0] - out[b]).abs().max().item())
report("4 rows independent", max(diffs) < 1e-3, f"(max diff vs single-row runs {max(diffs):.2e})")

# 5 — other split points
bad = []
for p in [0, T // 4, T // 2, T - 8]:
    try:
        o5, _ = run(ids, p)
        okp = tuple(o5.shape) in [(B, T, V), (B, T - max(p, 1) + 1, V)] and bool(torch.isfinite(o5).all())
        if not okp:
            bad.append((p, tuple(o5.shape)))
    except Exception as e:
        bad.append((p, repr(e)[:100]))
report("5 other split points", not bad, f"problems: {bad}" if bad else "")

# 6 — parameter budget
n = sum(q.numel() for q in m.parameters())
report("6 parameters within 50M-300M", 50e6 <= n <= 300e6, f"({n/1e6:.2f}M)")

# 7 — recurrent decoding near the end of the window
if hasattr(m, "prefill") and hasattr(m, "decode_step"):
    Bd, STEPS = 2, 32
    Tl = CTX if dev == "cuda" else min(CTX, 256)
    L = Tl - STEPS
    idl = torch.randint(5, V, (Bd, Tl))
    ref, offl = run(idl, L)
    errs, scale = [], ref.abs().max().item()
    with torch.no_grad():
        last, cache = m.prefill(idl[:, :L].to(dev), torch.full((Bd,), L, device=dev))
        errs.append((last.float() - ref[:, L - 1 - offl]).abs().max().item())
        for t in range(L, Tl - 1):
            last, cache = m.decode_step(idl[:, t:t + 1].to(dev), cache)
            errs.append((last.float() - ref[:, t - offl]).abs().max().item())
    worst = max(errs)
    report("7 recurrent decode matches forward", worst < 2e-3 * max(1.0, scale),
           f"(max diff {worst:.2e}, logit scale {scale:.2f}, {len(errs)} positions up to {Tl - 1})")
else:
    print("[SKIP] 7 recurrent decode (no prefill/decode_step): generation will recompute the whole "
          "sequence every token - much slower", flush=True)

# 8 — fp16 training step at full length (GPU)
if dev == "cuda":
    Bf = 2
    x = torch.randint(5, V, (Bf, CTX + 1), device=dev)
    Pf = (3 * CTX) // 4
    pl = torch.full((Bf,), Pf, device=dev)
    m.train()
    m.zero_grad(set_to_none=True)
    with torch.autocast("cuda", dtype=torch.float16):
        lg = m(x[:, :-1], pl)
        y = x[:, 1:].clone(); y[:, :Pf - 1] = -100
        y = y[:, y.size(1) - lg.size(1):]
        loss = F.cross_entropy(lg.float().reshape(-1, V), y.reshape(-1), ignore_index=-100)
    loss.backward()
    bad_grads = [nme for nme, q in m.named_parameters() if q.grad is not None and not torch.isfinite(q.grad).all()]
    no_grad = [nme for nme, q in m.named_parameters() if q.requires_grad and q.grad is None]
    finite = bool(torch.isfinite(lg).all()) and bool(torch.isfinite(loss))
    report("8 fp16 step finite at full ctx_len", finite and not bad_grads,
           f"(loss {loss.item():.3f}; non-finite grads: {bad_grads[:5]})" if bad_grads or not finite
           else f"(loss {loss.item():.3f} ~ ln V = {torch.log(torch.tensor(float(V))).item():.3f} at init)")
    if no_grad:
        print(f"  note: {len(no_grad)} parameters got no gradient (unused?): {no_grad[:5]}"
              " -> they are dead weight, and DDP needs find_unused=True", flush=True)
    m.eval(); m.zero_grad(set_to_none=True)
    with torch.no_grad():
        with torch.autocast("cuda", dtype=torch.float16):
            l16 = m(x[:, :-1], pl).float()
        l32 = m(x[:, :-1], pl).float()
    print(f"  info: |fp16 - fp32| logits max {(l16 - l32).abs().max().item():.3e} "
          f"(logit scale {l32.abs().max().item():.2f})", flush=True)
else:
    print("[SKIP] 8 fp16 step (no GPU)", flush=True)

print("\nALL CHECKS PASSED" if ok else "\nSOME CHECKS FAILED — fix model.py before training")
sys.exit(0 if ok else 1)

In [ ]:
# 7b — model checks (correctness) + memory/speed at your micro_batch
import importlib, sys, torch
sys.path.insert(0, "."); import model as M; importlib.reload(M)
assert not find_todos(MODEL) and TRAIN["prefix_mode"] != TODO, "fill MODEL and prefix_mode in CONFIG first"
V = _tok.get_vocab_size()
json.dump({"model": MODEL, "train": {"ctx_len": TRAIN["ctx_len"]}}, open("model_test_config.json", "w"))
import subprocess
_r = subprocess.run([sys.executable, "test_arch4_model.py", "model_test_config.json"], capture_output=True, text=True)
print(_r.stdout, _r.stderr[-3000:])
assert _r.returncode == 0, "model checks failed — see the FAIL lines / error above; fix model.py"

m = M.build_model(dict(MODEL, vocab_size=V, max_seq_len=TRAIN["ctx_len"])).cuda().train()
MB, CTX = TRAIN["micro_batch"], TRAIN["ctx_len"]
P = 0 if TRAIN["prefix_mode"] == "none" else int(round(TRAIN["prefix_frac_min"] * CTX))
torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t0 = None
for it in range(6):
    if it == 2: torch.cuda.synchronize(); t0 = time.time()
    xb = torch.randint(5, V, (MB, CTX + 1), device="cuda"); x, y = xb[:, :-1], xb[:, 1:].clone()
    if P > 1: y[:, :P - 1] = -100
    with torch.autocast("cuda", dtype=torch.float16):
        lg = m(x, torch.full((MB,), P, device="cuda"))
        yy = y[:, y.size(1) - lg.size(1):]
        loss = torch.nn.functional.cross_entropy(lg.float().reshape(-1, V), yy.reshape(-1), ignore_index=-100)
    loss.backward(); m.zero_grad(set_to_none=True)
torch.cuda.synchronize()
tps = 4 * MB * CTX / (time.time() - t0)
peak = torch.cuda.max_memory_allocated() / 1e9
train_h = SESSION_HOURS - RUN["reserve_minutes_after_train"] / 60 - (time.time() - SESSION_START) / 3600 - 0.25
est_tok = NGPU * tps * 0.85 * train_h * 3600
print(f"micro_batch {MB}: peak {peak:.1f} GB of {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} "
      f"(+~2 GB AdamW/DDP in training) | ~{tps/1e3:.1f}K tok/s per GPU (no optimizer)")
print(f"-> ~{NGPU*tps*0.85/1e3:.0f}K tok/s on {NGPU} GPU(s); ~{train_h:.1f} h of training -> ~{est_tok/1e9:.2f}B tokens")
del m, loss, xb, lg; torch.cuda.empty_cache()

In [ ]:
%%writefile dummy_model.py
"""dummy_model.py — PIPELINE TEST ONLY (used when RUN["smoke_test"] = True)."""
import torch.nn as nn


class _Dummy(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 64)
        self.out = nn.Linear(64, vocab_size)

    def forward(self, input_ids, prefix_lens):
        return self.out(self.emb(input_ids))


def build_model(model_cfg):
    return _Dummy(model_cfg["vocab_size"])

In [ ]:
%%writefile train.py
import contextlib, glob, json, math, os, sys, time

import numpy as np
import torch
import torch.distributed as dist
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel as DDP

CFG = json.load(open(sys.argv[1]))
RUN, MODEL_CFG, TR = CFG["run"], CFG["model"], CFG["train"]

if RUN.get("smoke_test"):
    import dummy_model as model_lib
else:
    import model as model_lib


# ----------------------------------------------------------------------------- setup
def setup_dist():
    if int(os.environ.get("WORLD_SIZE", "1")) > 1:
        dist.init_process_group("nccl")
        rank, world, local = dist.get_rank(), dist.get_world_size(), int(os.environ["LOCAL_RANK"])
    else:
        rank, world, local = 0, 1, 0
    torch.cuda.set_device(local)
    return rank, world, torch.device("cuda", local)


RANK, WORLD, DEV = setup_dist()
IS_MAIN = RANK == 0
DDP_ON = WORLD > 1


def log(*a):
    if IS_MAIN:
        print(*a, flush=True)


def bcast_float(x):
    t = torch.tensor([float(x)], device=DEV)
    if DDP_ON:
        dist.broadcast(t, 0)
    return t.item()


def allreduce_sum(*vals):
    t = torch.tensor([float(v) for v in vals], device=DEV, dtype=torch.float64)
    if DDP_ON:
        dist.all_reduce(t)
    return t.tolist()


torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

# ----------------------------------------------------------------------------- time budget
SESSION_START = RUN["session_start"]
SMOKE = bool(RUN.get("smoke_test"))
if SMOKE:
    TRAIN_DEADLINE = time.time() + RUN.get("smoke_train_minutes", 3) * 60
else:
    TRAIN_DEADLINE = (SESSION_START + RUN["session_hours"] * 3600
                      - RUN["reserve_minutes_after_train"] * 60)
log(f"Train deadline in {(TRAIN_DEADLINE - time.time())/3600:.2f} h")

# ----------------------------------------------------------------------------- data
META = json.load(open(os.path.join(RUN["token_dir"], "meta.json")))
CTX = int(TR["ctx_len"])


class WindowSampler:

    def __init__(self, split, mix, seed):
        files = sorted(glob.glob(os.path.join(RUN["token_dir"], f"{split}_*.bin")))
        assert files, f"no {split}_*.bin files in {RUN['token_dir']}"
        self.names, self.arrs = [], []
        for f in files:
            a = np.memmap(f, dtype=np.uint16, mode="r")
            if a.size > CTX + 2:
                self.names.append(os.path.basename(f)[len(split) + 1:-4])
                self.arrs.append(a)
        if mix == "proportional":
            w = np.array([a.size for a in self.arrs], dtype=np.float64)
        else:
            w = np.array([float(mix.get(n, 0.0)) for n in self.names])
        assert w.sum() > 0, f"mix gives zero weight to all of {self.names}"
        self.w = w / w.sum()
        self.rng = np.random.default_rng(seed)

    def window(self, i):
        a = self.arrs[i]
        s = int(self.rng.integers(0, a.size - CTX - 1))
        return np.asarray(a[s:s + CTX + 1], dtype=np.int64)

    def batch(self, B):
        idx = self.rng.choice(len(self.arrs), size=B, p=self.w)
        return torch.from_numpy(np.stack([self.window(i) for i in idx]))


def prefix_lengths(B, mode, fmin, fmax, rng):
    if mode == "none":
        return np.zeros(B, dtype=np.int64)
    lo = int(round(fmin * CTX))
    if mode == "fixed":
        return np.full(B, lo, dtype=np.int64)
    if mode == "random":
        hi = int(round(fmax * CTX))
        return rng.integers(lo, hi + 1, size=B).astype(np.int64)
    if mode == "batch_random":
        hi = int(round(fmax * CTX))
        return np.full(B, int(rng.integers(lo, hi + 1)), dtype=np.int64)
    raise ValueError(f"prefix_mode must be none|fixed|random|batch_random, got {mode!r}")


def make_xy(chunk, plen):
    x, y = chunk[:, :-1].clone(), chunk[:, 1:].clone()
    pos = torch.arange(CTX)[None, :]
    y[pos < (torch.from_numpy(plen)[:, None] - 1)] = -100
    return x, y, torch.from_numpy(plen)


train_sampler = WindowSampler("train", TR["mix"], seed=1234 + RANK * 7919)
train_prefix_rng = np.random.default_rng(99 + RANK)
log(f"Sources: {dict(zip(train_sampler.names, np.round(train_sampler.w, 3)))}")


VAL_PER_SOURCE = 8 if SMOKE else int(RUN.get("val_windows_per_source", 48))
val_sampler = WindowSampler("val", "proportional", seed=4321)
VAL = []
vp_rng = np.random.default_rng(7)
for si, name in enumerate(val_sampler.names):
    for _ in range(VAL_PER_SOURCE):
        c = torch.from_numpy(val_sampler.window(si))[None]
        pl = prefix_lengths(1, TR["val_prefix_mode"], TR["val_prefix_frac"], TR["val_prefix_frac"], vp_rng)
        x, y, p = make_xy(c, pl)
        VAL.append((name, x, y, p))
VAL_SHARD = VAL[RANK::WORLD]

# ----------------------------------------------------------------------------- model
MODEL_CFG.setdefault("vocab_size", META["vocab_size"])
MODEL_CFG.setdefault("max_seq_len", CTX)
raw_model = model_lib.build_model(MODEL_CFG).to(DEV)
n_params = sum(p.numel() for p in raw_model.parameters())
log(f"Parameters: {n_params/1e6:.2f}M")
if not SMOKE:
    assert 50e6 <= n_params <= 300e6, "Phase-1 rule: model must be 50M-300M parameters"
if TR.get("grad_ckpt") and hasattr(raw_model, "set_gradient_checkpointing"):
    raw_model.set_gradient_checkpointing(True)

decay, no_decay = [], []
for n, p in raw_model.named_parameters():
    if p.requires_grad:
        (decay if (p.dim() >= 2 or not TR["wd_exclude_1d"]) else no_decay).append(p)
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": TR["weight_decay"]},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=TR["lr_max"], betas=(TR["beta1"], TR["beta2"]), eps=1e-8, fused=True)
scaler = torch.amp.GradScaler("cuda", enabled=TR.get("amp_dtype", "fp16") == "fp16")
AMP_DTYPE = torch.float16 if TR.get("amp_dtype", "fp16") == "fp16" else torch.bfloat16

model = raw_model
if TR.get("compile"):
    model = torch.compile(model)
if DDP_ON:
    model = DDP(model, device_ids=[DEV.index], find_unused_parameters=bool(TR.get("find_unused", False)))


def unwrap():
    return raw_model


# ----------------------------------------------------------------------------- checkpoints
OUT = RUN["out_dir"]
os.makedirs(OUT, exist_ok=True)
NAME = RUN["run_name"]


def ckpt_files(d):
    return sorted(glob.glob(os.path.join(d, f"{NAME}_step*.pt")))


def save_ckpt(state, path):
    tmp = path + ".tmp"
    torch.save(state, tmp)
    os.replace(tmp, path)


def save_training_ckpt(step, tokens, total_steps, best_val):
    if not IS_MAIN:
        return
    path = os.path.join(OUT, f"{NAME}_step{step:07d}.pt")
    save_ckpt(dict(model=unwrap().state_dict(), optimizer=optimizer.state_dict(),
                   scaler=scaler.state_dict(), step=step, tokens=tokens,
                   total_steps=total_steps, best_val=best_val, cfg=CFG), path)
    for old in ckpt_files(OUT)[:-2]:
        os.remove(old)
    log(f"  saved {os.path.basename(path)}")


def save_model_only(tag, step, tokens, val):
    if IS_MAIN:
        save_ckpt(dict(model=unwrap().state_dict(), step=step, tokens=tokens, val_loss=val,
                       model_cfg=MODEL_CFG, n_params=n_params),
                  os.path.join(OUT, f"{NAME}_{tag}.pt"))


step, tokens, total_steps, best_val = 0, 0, None, float("inf")
RESUMED = False
cands = []
for d in [OUT] + list(RUN.get("resume_dirs", [])):
    cands += ckpt_files(d)
if cands:
    latest = max(cands, key=lambda p: int(p.split("_step")[-1][:-3]))
    ck = torch.load(latest, map_location=DEV, weights_only=False)
    unwrap().load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scaler.load_state_dict(ck["scaler"])
    step, tokens, total_steps, best_val = ck["step"], ck["tokens"], ck["total_steps"], ck["best_val"]
    log(f"Resumed from {latest} (step {step}, {tokens/1e6:.1f}M tokens)")
    RESUMED = True
    del ck

    train_sampler.rng = np.random.default_rng(1234 + RANK * 7919 + 1_000_003 * step)
    train_prefix_rng = np.random.default_rng(99 + RANK + 1_000_003 * step)
    log("Data sampler re-seeded for resume")

# ----------------------------------------------------------------------------- schedule
def lr_at(s, total):
    lr_max, lr_min, warm = TR["lr_max"], TR["lr_min"], TR["warmup_steps"]
    if s < warm:
        return lr_max * (s + 1) / warm
    if total is None:
        return lr_max
    t = min(1.0, (s - warm) / max(1, total - warm))
    kind = TR["schedule"]
    if kind == "cosine":
        return lr_min + (lr_max - lr_min) * 0.5 * (1 + math.cos(math.pi * t))
    if kind == "linear":
        return lr_max + (lr_min - lr_max) * t
    if kind == "constant":
        return lr_max
    if kind == "wsd":
        d = TR["decay_frac"]
        return lr_max if t < 1 - d else lr_max + (lr_min - lr_max) * (t - (1 - d)) / d
    raise ValueError(f"unknown schedule {kind!r}")


def align_targets(logits, y):
    L, T = logits.size(1), y.size(1)
    if L == T:
        return y
    if L > T or bool((y[:, :T - L] != -100).any()):
        raise ValueError(f"model returned {L} logit positions for {T} targets, but some dropped "
                         f"targets are scored: return at least T - prefix_len + 1 positions")
    return y[:, T - L:]


# ----------------------------------------------------------------------------- eval
@torch.no_grad()
def evaluate():
    model.eval()
    sums = {}
    for name, x, y, p in VAL_SHARD:
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            logits = unwrap()(x.to(DEV), p.to(DEV))
        y = align_targets(logits, y.to(DEV))
        l = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), y.reshape(-1),
                            ignore_index=-100, reduction="sum").item()
        n = (y != -100).sum().item()
        s = sums.setdefault(name, [0.0, 0])
        s[0] += l; s[1] += n
    names = val_sampler.names
    flat = []
    for n in names:
        flat += sums.get(n, [0.0, 0])
    flat = allreduce_sum(*flat)
    per = {n: flat[2 * i] / max(1, flat[2 * i + 1]) for i, n in enumerate(names)}
    tot = sum(flat[0::2]) / max(1, sum(flat[1::2]))
    model.train()
    return tot, per


# ----------------------------------------------------------------------------- train loop
MB, ACC = int(TR["micro_batch"]), int(TR["grad_accum"])
TOK_PER_STEP = MB * CTX * ACC * WORLD
log(f"Tokens/step: {TOK_PER_STEP:,}  (micro {MB} x ctx {CTX} x accum {ACC} x gpus {WORLD})")

csv_path = os.path.join(OUT, f"{NAME}_log.csv")
if IS_MAIN and not os.path.exists(csv_path):
    open(csv_path, "w").write("step,tokens,train_loss,lr,tok_per_s,elapsed_h,val_loss,val_per_source,"
                               "grad_norm_max,amp_scale,skipped_steps\n")


def csv(row):
    if IS_MAIN:
        open(csv_path, "a").write(row + "\n")


CALIB_AFTER = 10
CALIB_STEPS = 30
calib_t0 = None
run_t0 = time.time()
last_ckpt = time.time()
last_val = time.time()
win_t0, win_tok = time.time(), 0
model.train()
optimizer.zero_grad(set_to_none=True)
steps_this_run = 0


gn_max_win = torch.zeros((), device=DEV)
skipped_total, skipped_win = 0, 0

while True:

    loss_acc = torch.zeros((), device=DEV)
    for micro in range(ACC):
        chunk = train_sampler.batch(MB)
        pl = prefix_lengths(MB, TR["prefix_mode"], TR.get("prefix_frac_min", 0.0),
                            TR.get("prefix_frac_max", 0.0), train_prefix_rng)
        x, y, p = make_xy(chunk, pl)
        x, y, p = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True), p.to(DEV, non_blocking=True)
        sync_ctx = model.no_sync() if (DDP_ON and micro < ACC - 1) else contextlib.nullcontext()
        with sync_ctx:
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                logits = model(x, p)
                yy = align_targets(logits, y)
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), yy.reshape(-1),
                                       ignore_index=-100)
            scaler.scale(loss / ACC).backward()
        loss_acc += loss.detach() / ACC

    lr = lr_at(step, total_steps)
    for g in optimizer.param_groups:
        g["lr"] = lr
    scaler.unscale_(optimizer)
    gn = torch.nn.utils.clip_grad_norm_(raw_model.parameters(), TR["grad_clip"])
    gn_max_win = torch.maximum(gn_max_win, torch.nan_to_num(gn.detach().float(), nan=float("inf")))
    scale_before = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()
    if scaler.is_enabled() and scaler.get_scale() < scale_before:
        skipped_total += 1; skipped_win += 1
    optimizer.zero_grad(set_to_none=True)
    step += 1
    steps_this_run += 1
    tokens += TOK_PER_STEP
    win_tok += TOK_PER_STEP


    if steps_this_run == CALIB_AFTER:
        torch.cuda.synchronize(); calib_t0 = time.time()
    if steps_this_run == CALIB_AFTER + CALIB_STEPS and (total_steps is None or RESUMED):
        torch.cuda.synchronize()
        sec_per_step = (time.time() - calib_t0) / CALIB_STEPS
        overhead = 1.0 - RUN.get("eval_ckpt_overhead_frac", 0.04)
        est = step + int((TRAIN_DEADLINE - time.time()) / sec_per_step * overhead)
        if TR.get("max_tokens"):
            est = min(est, int(TR["max_tokens"]) // TOK_PER_STEP)
        est = int(bcast_float(est))
        if RESUMED and total_steps is not None:
            if est < total_steps:
                log(f"Resume: only time for {est} of {total_steps} scheduled steps -> schedule shortened so LR still decays")
                total_steps = est
        else:
            total_steps = est
        log(f"Measured {sec_per_step:.2f}s/step ({TOK_PER_STEP/sec_per_step/1e3:.1f}K tok/s). "
            f"Schedule sized to {total_steps} steps = {total_steps*TOK_PER_STEP/1e6:.0f}M tokens.")

    now = time.time()
    if step % TR.get("log_every_steps", 20) == 0:
        l = allreduce_sum(loss_acc.item())[0] / WORLD
        tps = win_tok / max(1e-9, now - win_t0)
        gmax = gn_max_win.item()
        sc = scaler.get_scale() if scaler.is_enabled() else 1.0
        log(f"step {step:6d} | loss {l:.4f} | lr {lr:.2e} | {tokens/1e6:8.1f}M tok | "
            f"{tps/1e3:5.1f}K tok/s | gnorm {gmax:6.2f} | scale {sc:.0f} | skip {skipped_win}/{skipped_total} | "
            f"left {(TRAIN_DEADLINE-now)/3600:.2f}h")
        csv(f"{step},{tokens},{l:.5f},{lr:.3e},{tps:.0f},{(now-run_t0)/3600:.3f},,,{gmax:.4f},{sc:.0f},{skipped_total}")
        win_t0, win_tok = now, 0
        gn_max_win.zero_(); skipped_win = 0


    flags = torch.tensor([
        float((now >= TRAIN_DEADLINE) or (total_steps is not None and step >= total_steps)),
        float(now - last_val >= TR.get("val_every_min", 30) * 60),
        float(now - last_ckpt >= TR.get("ckpt_every_min", 20) * 60)], device=DEV)
    if DDP_ON:
        dist.broadcast(flags, 0)
    stop, do_val, do_ckpt = (bool(f) for f in flags.tolist())

    if stop or do_val:
        v, per = evaluate()
        log(f"  [val] loss {v:.4f} ppl {math.exp(min(v, 20)):.2f} | " +
            " ".join(f"{k}={x:.3f}" for k, x in per.items()))
        csv(f"{step},{tokens},,{lr:.3e},,{(now-run_t0)/3600:.3f},{v:.5f},"
            + ";".join(f"{k}={x:.4f}" for k, x in per.items()) + ",,,")
        if v < best_val:
            best_val = v
            save_model_only("best", step, tokens, v)
            log(f"  new best -> {NAME}_best.pt")
        last_val = time.time()

    if stop or do_ckpt:
        save_training_ckpt(step, tokens, total_steps, best_val)
        last_ckpt = time.time()

    if stop:
        break

save_model_only("final", step, tokens, best_val)
log(f"Training finished: {step} steps, {tokens/1e6:.1f}M tokens, best val {best_val:.4f}")
if DDP_ON:
    dist.barrier()
    dist.destroy_process_group()

In [ ]:
%%writefile generate.py
import glob, json, os, re, sys, time, zipfile

import torch
import torch.distributed as dist
import torch.nn.functional as F

CFG = json.load(open(sys.argv[1]))
IN_PATH, OUT_PREFIX = sys.argv[2], sys.argv[3]
RUN, MODEL_CFG, GEN = CFG["run"], CFG["model"], CFG["gen"]
TARGET_WORDS = 128

if RUN.get("smoke_test"):
    import dummy_model as model_lib
else:
    import model as model_lib

if int(os.environ.get("WORLD_SIZE", "1")) > 1:
    LOCAL = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(LOCAL)
    try:
        dist.init_process_group("nccl", device_id=torch.device("cuda", LOCAL))
    except TypeError:
        dist.init_process_group("nccl")
    RANK, WORLD = dist.get_rank(), dist.get_world_size()
else:
    RANK, WORLD, LOCAL = 0, 1, 0
torch.cuda.set_device(LOCAL)
DEV = torch.device("cuda", LOCAL)
torch.manual_seed(int(GEN.get("seed", 0)) + RANK)


def log(*a):
    if RANK == 0:
        print(*a, flush=True)


# ----------------------------------------------------------------------------- tokenizer + model
from tokenizers import ByteLevelBPETokenizer

tok = ByteLevelBPETokenizer(vocab=os.path.join(RUN["tokenizer_dir"], "vocab.json"),
                            merges=os.path.join(RUN["tokenizer_dir"], "merges.txt"))
CTX = int(CFG["train"]["ctx_len"])
MODEL_CFG.setdefault("vocab_size", tok.get_vocab_size())
MODEL_CFG.setdefault("max_seq_len", CTX)

ck_choice = GEN["checkpoint"]
ck_path = ck_choice if os.path.exists(ck_choice) else os.path.join(RUN["out_dir"], f"{RUN['run_name']}_{ck_choice}.pt")
model = model_lib.build_model(MODEL_CFG).to(DEV)
ck = torch.load(ck_path, map_location=DEV, weights_only=False)
model.load_state_dict(ck["model"])
model.eval()
log(f"Loaded {ck_path} (step {ck.get('step')}, val {ck.get('val_loss')})")
del ck
HAS_CACHE = hasattr(model, "prefill") and hasattr(model, "decode_step")
log("Decoding with KV cache (prefill/decode_step)" if HAS_CACHE else
    "No prefill/decode_step on the model: recomputing the full sequence each step (slower)")

# ----------------------------------------------------------------------------- sampling utils
MAX_NEW = int(GEN["max_new_tokens"])
PROMPT_MAX = CTX - MAX_NEW
assert PROMPT_MAX > 64, "ctx_len too small for max_new_tokens"
BAN = [tok.token_to_id(t) for t in ("<s>", "</s>", "<unk>", "<pad>", "<mask>") if tok.token_to_id(t) is not None]


class NgramBlocker:

    def __init__(self, n, prompt_ids, scope):
        self.n, self.seen = n, {}
        self.hist = list(prompt_ids[-(n - 1):]) if (n and scope != "all") else []
        if n and scope == "all":
            for i in range(len(prompt_ids) - n + 1):
                self.seen.setdefault(tuple(prompt_ids[i:i + n - 1]), set()).add(prompt_ids[i + n - 1])
            self.hist = list(prompt_ids[-(n - 1):])
        self.scope = scope
        self.gen_count = 0

    def banned(self):
        if not self.n or len(self.hist) < self.n - 1:
            return ()
        return self.seen.get(tuple(self.hist[-(self.n - 1):]), ())

    def push(self, t):
        if self.n:
            key = tuple(self.hist[-(self.n - 1):])

            if len(key) == self.n - 1 and (self.scope == "all" or self.gen_count >= self.n - 1):
                self.seen.setdefault(key, set()).add(t)
            self.hist.append(t)
            self.gen_count += 1


def pick_next(logits, gen_tokens, blockers):
    if BAN:
        logits[:, BAN] = -float("inf")
    rp = GEN["rep_penalty"]
    if rp and rp != 1.0 and gen_tokens.numel():
        sc = torch.gather(logits, 1, gen_tokens)
        logits.scatter_(1, gen_tokens, torch.where(sc < 0, sc * rp, sc / rp))
    for b, bl in enumerate(blockers):
        bad = bl.banned()
        if bad:
            logits[b, list(bad)] = -float("inf")
    T = GEN["temperature"]
    if not T:
        return logits.argmax(-1)
    logits = logits / T
    k = GEN.get("top_k") or 0
    if k:
        kth = torch.topk(logits, k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    p = GEN.get("top_p") or 1.0
    probs = F.softmax(logits, -1)
    if p < 1.0:
        sp, si = probs.sort(-1, descending=True)
        sp[(sp.cumsum(-1) - sp) > p] = 0.0
        probs = torch.zeros_like(probs).scatter_(1, si, sp)
    probs = probs / probs.sum(-1, keepdim=True)
    return torch.multinomial(probs, 1).squeeze(1)


def clean(text):
    return re.sub(r"\s+", " ", text).strip()


@torch.inference_mode()
def generate_batch(prompts_ids):
    B, L = len(prompts_ids), len(prompts_ids[0])
    ids = torch.tensor(prompts_ids, dtype=torch.long, device=DEV)
    plen = torch.full((B,), L, dtype=torch.long, device=DEV)
    blockers = [NgramBlocker(GEN.get("no_repeat_ngram") or 0, p, GEN.get("ngram_scope", "generated"))
                for p in prompts_ids]
    gen = torch.empty((B, 0), dtype=torch.long, device=DEV)
    done = torch.zeros(B, dtype=torch.bool, device=DEV)
    with torch.autocast("cuda", dtype=torch.float16):
        if HAS_CACHE:
            logits, cache = model.prefill(ids, plen)
        else:
            logits = model(ids, plen)[:, -1]
    for step in range(MAX_NEW):
        nxt = pick_next(logits.float(), gen, blockers)
        gen = torch.cat([gen, nxt[:, None]], 1)
        nl = nxt.tolist()
        for b, bl in enumerate(blockers):
            bl.push(nl[b])
        if (step + 1) % 16 == 0:
            for b in range(B):
                if not done[b] and len(tok.decode(gen[b].tolist()).split()) > TARGET_WORDS:
                    done[b] = True
            if bool(done.all()):
                break
        if step == MAX_NEW - 1:
            break
        with torch.autocast("cuda", dtype=torch.float16):
            if HAS_CACHE:
                logits, cache = model.decode_step(nxt[:, None], cache)
            else:
                logits = model(torch.cat([ids, gen], 1), plen)[:, -1]
    return [clean(tok.decode(g.tolist())) for g in gen]


# ----------------------------------------------------------------------------- run
data = json.load(open(IN_PATH, encoding="utf-8"))
all_ids = list(data.keys())
if RUN.get("smoke_test"):
    all_ids = all_ids[:16]
mine = all_ids[RANK::WORLD]
enc = {k: tok.encode(data[k]["text"]).ids[-PROMPT_MAX:] for k in mine}
order = sorted(mine, key=lambda k: len(enc[k]))
BS = int(GEN["batch_size"])
out, dropped = {}, 0
t0 = time.time()
for i in range(0, len(order), BS):
    keys = order[i:i + BS]
    m = min(len(enc[k]) for k in keys)
    dropped += sum(len(enc[k]) - m for k in keys)
    texts = generate_batch([enc[k][-m:] for k in keys])
    for k, t in zip(keys, texts):
        out[k] = t
    if RANK == 0 and (i // BS) % 5 == 0:
        el = time.time() - t0
        print(f"  rank0 {i+len(keys)}/{len(order)} prompts | {el/60:.1f} min | "
              f"eta {el/(i+len(keys))*(len(order)-i-len(keys))/60:.1f} min", flush=True)


del model
torch.cuda.empty_cache()

part = f"{OUT_PREFIX}.part{RANK}.json"
json.dump(out, open(part, "w", encoding="utf-8"), ensure_ascii=False)
log(f"Context tokens trimmed for batching (rank0): {dropped} total")
if WORLD > 1:
    dist.barrier()

if RANK == 0:
    merged = {}
    for pf in sorted(glob.glob(f"{OUT_PREFIX}.part*.json")):
        merged.update(json.load(open(pf, encoding="utf-8")))
        os.remove(pf)
    preds, short = {}, 0
    for k in all_ids:
        w = merged.get(k, "").split()
        if len(w) < TARGET_WORDS:
            short += 1
        preds[k] = {"text": " ".join(w[:TARGET_WORDS])}
    pred_path = f"{OUT_PREFIX}_prediction.json"
    json.dump(preds, open(pred_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    zip_path = f"{OUT_PREFIX}_submission.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path, arcname="prediction.json")
    print(f"Wrote {pred_path} and {zip_path} | {len(preds)} ids | {short} under {TARGET_WORDS} words | "
          f"{(time.time()-t0)/60:.1f} min", flush=True)
if WORLD > 1:
    dist.destroy_process_group()


## Smoke test (≈5 min) — run before every real run
Uses `dummy_model.py` and tiny pipeline-test settings for anything still `TODO` (**not** recommendations for your model).
Checks data loading, DDP, checkpoint save/resume, validation, generation, zip creation.

In [ ]:
# 12 — smoke test
import copy
smoke = copy.deepcopy(CFG)
PIPELINE_TEST_ONLY = dict(ctx_len=512, micro_batch=4, grad_accum=2, lr_max=1e-3, lr_min=1e-4, warmup_steps=10,
                          prefix_mode="random", prefix_frac_min=0.25, prefix_frac_max=0.75)
smoke["train"].update(PIPELINE_TEST_ONLY)
smoke["gen"].update(checkpoint="final", batch_size=8)
SMOKE_DIR = f"{WORK}/smoke"
smoke["run"].update(smoke_test=True, smoke_train_minutes=2, out_dir=SMOKE_DIR, run_name="smoke",
                    val_windows_per_source=8, resume_dirs=[])
smoke["train"].update(val_every_min=1, ckpt_every_min=1)
json.dump(smoke, open("smoke_config.json", "w"))
shutil.rmtree(SMOKE_DIR, ignore_errors=True)
!timeout -k 30 600 torchrun --standalone --nproc_per_node={NGPU} train.py smoke_config.json
print("---- resume check (should say 'Resumed from') ----")
smoke["run"]["smoke_train_minutes"] = 0.5; json.dump(smoke, open("smoke_config.json", "w"))
!timeout -k 30 600 torchrun --standalone --nproc_per_node={NGPU} train.py smoke_config.json
!timeout -k 30 600 torchrun --standalone --nproc_per_node={NGPU} generate.py smoke_config.json {TOKEN_DIR}/local_test.json {SMOKE_DIR}/local
_ok = os.path.exists(f"{SMOKE_DIR}/smoke_final.pt") and os.path.exists(f"{SMOKE_DIR}/local_submission.zip")
assert _ok, "SMOKE TEST FAILED — read the torchrun output above; the real run was not started"
shutil.rmtree(SMOKE_DIR, ignore_errors=True)
print("SMOKE TEST PASSED")

## Real run
Trains until `session_hours − reserve_minutes_after_train`, keeping `arch4_best.pt` (lowest validation loss, checked every 15 min) and `arch4_final.pt`.
Each log line shows `gnorm` (largest gradient norm before clipping in the window), the fp16 loss `scale` and `skip` (optimizer steps skipped for inf/nan gradients: this window / total).
A climbing `skip`, a scale that keeps halving, or `gnorm` jumping by 10× with the loss rising = training is going unstable.

In [ ]:
# 14 — train (your model.py, your config)
assert not find_todos(CFG), "fill every TODO first"
json.dump(CFG, open("run_config.json", "w"), indent=1)
_cap = int(SESSION_START + RUN["session_hours"] * 3600 - RUN["reserve_minutes_after_train"] * 60 - time.time()) + 1200
!timeout -k 60 {_cap} torchrun --standalone --nproc_per_node={NGPU} train.py run_config.json
_best = os.path.join(OUT_DIR, f"{RUN_NAME}_best.pt")
_final = os.path.join(OUT_DIR, f"{RUN_NAME}_final.pt")
assert os.path.exists(_best), "TRAINING FAILED — no checkpoint at all; see output above"
if not os.path.exists(_final):
    print("WARNING: training did not finish cleanly (no final.pt) — generating from best.pt")
    GEN["checkpoint"] = CFG["gen"]["checkpoint"] = "best"; json.dump(CFG, open("run_config.json", "w"), indent=1)
print("training finished:", _best)


def minutes_left():
    return (SESSION_START + RUN["session_hours"] * 3600 - time.time()) / 60

In [ ]:
# 15 — local evaluation on held-out pairs: ROUGE-L + BERTScore F1 (roberta-large, as on Codabench)
DO_LOCAL = minutes_left() > 30
print(f"{minutes_left():.0f} min left | local eval: {DO_LOCAL}")


def rouge_l_f1(a, b):
    a, b = a.split(), b.split()
    if not a or not b: return 0.0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[-1]))
        prev = cur
    l = prev[-1]
    if l == 0: return 0.0
    p, r = l / len(a), l / len(b)
    return 2 * p * r / (p + r)


def bertscore_f1(cands, refs):
    try:
        from bert_score import score as _bs
        _, _, f = _bs(cands, refs, lang="en", batch_size=32, verbose=False)
        torch.cuda.empty_cache()
        return float(f.mean())
    except Exception as e:
        print("BERTScore unavailable:", repr(e)[:120]); return None


def score_file(pred_path):
    p = json.load(open(pred_path)); ks = list(ref)
    rl = sum(rouge_l_f1(p[k]["text"], ref[k]["text"]) for k in ks) / len(ks)
    bs = bertscore_f1([p[k]["text"] for k in ks], [ref[k]["text"] for k in ks])
    return rl, bs


if DO_LOCAL:
    !timeout -k 30 1500 torchrun --standalone --nproc_per_node={NGPU} generate.py run_config.json {TOKEN_DIR}/local_test.json {OUT_DIR}/local
    if not os.path.exists(f"{OUT_DIR}/local_prediction.json"):
        print("local generation failed — skipping local eval/sweep, going straight to the test set")
        DO_LOCAL = False
if DO_LOCAL:
    ref = json.load(open(f"{TOKEN_DIR}/local_ref.json"))
    origin = json.load(open(f"{TOKEN_DIR}/local_origin.json"))
    pred = json.load(open(f"{OUT_DIR}/local_prediction.json"))
    by_src = {}
    for k in ref:
        by_src.setdefault(origin[k], []).append(rouge_l_f1(pred[k]["text"], ref[k]["text"]))
    BASE_RL, BASE_BS = score_file(f"{OUT_DIR}/local_prediction.json")
    print(f"local (n={len(ref)}): ROUGE-L {BASE_RL:.4f} | BERTScore F1 {BASE_BS if BASE_BS is None else round(BASE_BS, 4)}")
    for s, v in by_src.items(): print(f"  ROUGE-L {s:15s} {sum(v)/len(v):.4f}  (n={len(v)})")

In [ ]:
# 15b — decoding sweep on the local held-out pairs; the best local BERTScore (ROUGE-L if BERTScore is

if not DO_LOCAL:
    print("skipping decoding sweep (no time); test generation uses GEN from the config cell")
else:
    import copy
    CANDS = {
        "greedy_nr3_rp1.1": dict(temperature=0, top_k=0, top_p=1.0, rep_penalty=1.1, no_repeat_ngram=3),
        "greedy_rp1.2":     dict(temperature=0, top_k=0, top_p=1.0, rep_penalty=1.2, no_repeat_ngram=0),
        "sample_t0.7":      dict(temperature=0.7, top_k=50, top_p=0.9, rep_penalty=1.1, no_repeat_ngram=4),
    }
    results = {"config (cell 3)": (BASE_RL, BASE_BS, {})}
    for name, over in CANDS.items():
        if minutes_left() < 35:
            print(f"skipping {name}: {minutes_left():.0f} min left, keeping time for the test set"); break
        c = copy.deepcopy(CFG); c["gen"].update(over); json.dump(c, open(f"gen_{name}.json", "w"))
        !timeout -k 30 1200 torchrun --standalone --nproc_per_node={NGPU} generate.py gen_{name}.json {TOKEN_DIR}/local_test.json {OUT_DIR}/sweep_{name}
        if not os.path.exists(f"{OUT_DIR}/sweep_{name}_prediction.json"):
            print(f"{name} failed, skipped"); continue
        rl, bs = score_file(f"{OUT_DIR}/sweep_{name}_prediction.json")
        results[name] = (rl, bs, over)
    use_bs = all(v[1] is not None for v in results.values())
    key = (lambda kv: -kv[1][1]) if use_bs else (lambda kv: -kv[1][0])
    for k, (rl, bs, _) in sorted(results.items(), key=key):
        print(f"  {k:20s} BERTScore {bs if bs is None else round(bs, 4)} | ROUGE-L {rl:.4f}")
    best = sorted(results.items(), key=key)[0][0]
    CFG["gen"].update(results[best][2]); GEN = CFG["gen"]
    json.dump(CFG, open("run_config.json", "w"), indent=1)
    print("test generation will use:", best, "(selected by", "BERTScore)" if use_bs else "ROUGE-L)",
          {k: GEN[k] for k in ("temperature", "top_k", "top_p", "rep_penalty", "no_repeat_ngram")})

In [ ]:
# 16 — official test set -> submission zip
_cap = max(600, int(SESSION_START + RUN['session_hours'] * 3600 - time.time()) - 120)
!timeout -k 30 {_cap} torchrun --standalone --nproc_per_node={NGPU} generate.py run_config.json {TEST_PATH} {OUT_DIR}/test
_zip = os.path.join(OUT_DIR, "test_submission.zip")
assert os.path.exists(_zip), "TEST GENERATION FAILED — see output above"
SUBMISSION = f"{WORK}/submission.zip"
shutil.copy(_zip, SUBMISSION)
shutil.copy(os.path.join(OUT_DIR, "test_prediction.json"), f"{WORK}/prediction.json")
if ON_COLAB:
    shutil.copy(_zip, f"{DRIVE_DIR}/submission.zip")
    print("also saved to Drive: MyDrive/arch4/submission.zip")
print("ready:", SUBMISSION)

In [ ]:
# 17 — validate the zip before uploading + look at the short outputs
import zipfile
def validate_zip(zip_path, test_path):
    test = json.load(open(test_path, encoding="utf-8"))
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        assert names == ["prediction.json"], f"zip must contain only prediction.json at the root, has {names}"
        pred = json.loads(z.read("prediction.json").decode("utf-8"))
    missing, extra = set(test) - set(pred), set(pred) - set(test)
    assert not missing, f"{len(missing)} test IDs missing, e.g. {list(missing)[:5]}"
    assert not extra, f"{len(extra)} unknown IDs, e.g. {list(extra)[:5]}"
    bad = [k for k, v in pred.items() if not isinstance(v, dict) or not isinstance(v.get("text"), str)]
    assert not bad, f"bad entries: {bad[:5]}"
    wc = {k: len(v["text"].split()) for k, v in pred.items()}
    empty = sum(w == 0 for w in wc.values())
    leaked = sum(pred[k]["text"][:200] in test[k]["text"] for k in pred if pred[k]["text"])
    print(f"OK: {len(pred)} ids | words min/mean/max {min(wc.values())}/{sum(wc.values())/len(wc):.1f}/{max(wc.values())} | "
          f"empty {empty} | outputs that copy prompt text {leaked}")
    short = sorted((w, k) for k, w in wc.items() if w < 128)
    print(f"{len(short)} outputs under 128 words. Shortest (these used all max_new_tokens on text with few spaces):")
    for w, k in short[:5]:
        print(f"  id {k} ({w} words): {pred[k]['text'][:160]!r}")
validate_zip(SUBMISSION, TEST_PATH)
if ON_KAGGLE:
    from IPython.display import FileLink; display(FileLink("submission.zip"))

In [ ]:
# 18 — training curves for the report (loss, throughput, gradient norm, fp16 loss scale)
import pandas as pd, matplotlib.pyplot as plt
lg = pd.read_csv(f"{OUT_DIR}/{RUN_NAME}_log.csv")
tr, va = lg[lg.train_loss.notna()], lg[lg.val_loss.notna()]
has_health = "grad_norm_max" in lg.columns
fig, ax = plt.subplots(1, 3 if has_health else 2, figsize=(16 if has_health else 12, 4))
ax[0].plot(tr.tokens / 1e6, tr.train_loss, lw=0.8, label="train"); ax[0].plot(va.tokens / 1e6, va.val_loss, "o-", label="val")
ax[0].set_xlabel("tokens (M)"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title(RUN_NAME)
ax[1].plot(tr.tokens / 1e6, tr.tok_per_s / 1e3); ax[1].set_xlabel("tokens (M)"); ax[1].set_ylabel("K tokens/s")
if has_health:
    ax[2].semilogy(tr.tokens / 1e6, tr.grad_norm_max, lw=0.8, label="grad norm (max/window)")
    ax[2].semilogy(tr.tokens / 1e6, tr.amp_scale, lw=0.8, label="fp16 loss scale")
    ax[2].set_xlabel("tokens (M)"); ax[2].legend(); ax[2].set_title(f"skipped steps: {int(tr.skipped_steps.max())}")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/{RUN_NAME}_curves.png", dpi=120); plt.show()

In [ ]:
# 19 — tidy outputs (Kaggle: small output mounts reliably; Colab: Drive space)
for f in glob.glob(f"{OUT_DIR}/sweep_*") + glob.glob(f"{WORK}/gen_*.json"):
    os.remove(f)
_steps = sorted(glob.glob(f"{OUT_DIR}/{RUN_NAME}_step*.pt"), key=lambda q: int(q.split("_step")[-1][:-3]))
for f in _steps[:-1]:
    os.remove(f)
tot = 0
for root, _, files in os.walk(OUT_DIR):
    for f in files:
        q = os.path.join(root, f); s = os.path.getsize(q); tot += s
        if s > 50e6: print(f"{s/1e9:6.2f} GB  {q}")
print(f"{OUT_DIR}: {tot/1e9:.2f} GB  | keep {RUN_NAME}_best.pt for Phase 2")